In [20]:
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
import re #regular expression
from nltk.corpus import stopwords # nltk is natural language toolkit and stopwords are the words not adding much value to the para
from nltk.stem.porter import PorterStemmer 

from sklearn.preprocessing import StandardScaler,Normalizer,OneHotEncoder,LabelEncoder,OrdinalEncoder,MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer # convert texts to feature vectors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression,Lasso,Ridge,ElasticNet
from sklearn.tree import DecisionTreeClassifier,plot_tree
from mlxtend.plotting import plot_decision_regions
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier,AdaBoostRegressor,BaggingClassifier,BaggingRegressor,GradientBoostingClassifier,GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error,mean_squared_error,mean_absolute_error,accuracy_score,precision_score,r2_score,recall_score,confusion_matrix,classification_report
from xgboost import XGBClassifier,XGBRegressor

In [21]:
import nltk 
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prathamsharma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [22]:
#examples of stopwords 
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [23]:
df = pd.read_csv("/Users/prathamsharma/Desktop/Projects/Fake News Prediction/news.csv")

In [24]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [25]:
df.shape

(6335, 4)

In [26]:
df.isnull().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [27]:
x = df.drop("label",axis=1)
y=df["label"]
print(x.shape,y.shape)

(6335, 3) (6335,)


In [28]:
# stemmiing is a method to reduce several words to its root word
port_stem = PorterStemmer()

def stemming(content):
    stemmed_content = re.sub("[^a-zA-Z]"," ",content) # only need aphlabest drom a-z and A-Z and if any anything extra replace with " " and do all these to the data in content
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words("english")]
    stemmed_content = " ".join(stemmed_content)
    return stemmed_content

In [29]:
df["text"] = df["text"].apply(stemming)

In [30]:
print(df["text"])

0       daniel greenfield shillman journal fellow free...
1       googl pinterest digg linkedin reddit stumbleup...
2       u secretari state john f kerri said monday sto...
3       kayde king kaydeek novemb lesson tonight dem l...
4       primari day new york front runner hillari clin...
                              ...                        
6330    state depart told republican nation committe c...
6331    p pb stand plutocrat pentagon post oct wikimed...
6332    anti trump protest tool oligarchi reform alway...
6333    addi ababa ethiopia presid obama conven meet l...
6334    jeb bush suddenli attack trump matter jeb bush...
Name: text, Length: 6335, dtype: object


SEPERATING DATA AND LABEL

In [31]:
x= df["text"].values
y=df["label"].values

In [32]:
print(x.shape,y.shape)

(6335,) (6335,)


In [33]:
print(x)

['daniel greenfield shillman journal fellow freedom center new york writer focus radic islam final stretch elect hillari rodham clinton gone war fbi word unpreced thrown around often elect ought retir still unpreced nomine major polit parti go war fbi exactli hillari peopl done coma patient wake watch hour cnn hospit bed would assum fbi director jame comey hillari oppon elect fbi attack everyon obama cnn hillari peopl circul letter attack comey current media hit piec lambast target trump surpris clinton alli start run attack ad fbi fbi leadership warn entir left wing establish form lynch mob continu go hillari fbi credibl attack media democrat preemptiv head result investig clinton foundat hillari clinton covert struggl fbi agent obama doj peopl gone explos public new york time compar comey j edgar hoover bizarr headlin jame comey role recal hoover fbi fairli practic admit front spout nonsens boston globe publish column call comey resign outdon time editori claim scandal realli attack 

In [34]:
print(y)

['FAKE' 'FAKE' 'REAL' ... 'FAKE' 'REAL' 'REAL']


CONVERTING TEXTUAL DATA TO NUMERICAL DATA

In [35]:
vectorizer = TfidfVectorizer()
vectorizer.fit(x)

x=vectorizer.transform(x)
print(x)

  (0, 106)	0.02761351227547292
  (0, 212)	0.023325671822305483
  (0, 271)	0.054392547007153
  (0, 328)	0.060358339691341766
  (0, 357)	0.013921754610837712
  (0, 451)	0.020578335403474236
  (0, 578)	0.07906058794712593
  (0, 617)	0.019850889079147057
  (0, 620)	0.01867769280554434
  (0, 622)	0.044611051373187424
  (0, 648)	0.015493583340569357
  (0, 680)	0.019224695171899866
  (0, 978)	0.0267316264615063
  (0, 999)	0.020738114075644642
  (0, 1022)	0.0371485992478585
  (0, 1033)	0.014808690907292291
  (0, 1085)	0.028815013247480384
  (0, 1089)	0.018102417756284926
  (0, 1197)	0.022505695388032227
  (0, 1206)	0.010675511636980443
  (0, 1549)	0.02507693986049084
  (0, 1665)	0.025967702221880915
  (0, 1713)	0.041128098634750046
  (0, 1744)	0.027618563562476325
  (0, 1783)	0.018872240981032625
  :	:
  (6334, 39650)	0.023985120164234364
  (6334, 39971)	0.0407182753213865
  (6334, 40086)	0.03890236771964674
  (6334, 40158)	0.04886237540126324
  (6334, 40783)	0.058599612687679964
  (6334, 4130

SPLITTING TRAINING AND TESTING DATA

In [36]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=2,stratify=y)
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(5068, 43733) (1267, 43733) (5068,) (1267,)


MODEL TRAINING

In [37]:
logreg = LogisticRegression()

In [38]:
logreg.fit(x_train,y_train)

y_pred_train = logreg.predict(x_train)
y_pred_test = logreg.predict(x_test)

accuracy_training = accuracy_score(y_train,y_pred_train)
accuracy_testing = accuracy_score(y_test,y_pred_test)

print("ACCURACY SCORE FOR TRAINING DATA=",accuracy_training)
print("ACCURACY SCORE FOR TESTING DATA=",accuracy_testing)

ACCURACY SCORE FOR TRAINING DATA= 0.9518547750591949
ACCURACY SCORE FOR TESTING DATA= 0.9163378058405682
